# Store Sales: Time Series Forecasting
**Kaggle Getting Started competition**: forecast 16 days of daily sales for Corporación Favorita, a large Ecuadorian
grocery chain.

- **Task:** predict `sales` for 1,782 series (54 stores × 33 product families) on every day from 16 to 31 August 2017.
- **Metric:** RMSLE, root mean squared logarithmic error, $\sqrt{\text{mean}\,(\log(1+\hat y) - \log(1+y))^2}$. It scores
  relative errors, so a small family such as BOOKS counts as much as GROCERY I. Training on `log1p(sales)` with
  squared-error loss optimises it directly.
- **Data:** 3.0M daily rows from 1 January 2013 to 15 August 2017, plus promotions (known for the test days too),
  store metadata, holidays and events, oil prices and store transactions.
- **Leaderboard:** a permanent competition whose leaderboard rolls (entries drop off after about two months). In
  October 2026 the top score was 0.3717, the top 10% 0.389 and the median 0.469. The best-known public notebook, a
  global LightGBM built with Darts, scores 0.380.
- **Approach:** direct multi-horizon gradient boosting, the method that won Kaggle's original 2018 Favorita
  competition:
  1. reshape the data into a series × day matrix;
  2. snapshot the history on many past Wednesdays (the test window starts on a Wednesday). Each snapshot gives one
     training row per series, with features computed only from the days before it;
  3. train **16 LightGBM models, one per forecast day**. No forecast is fed back in as an input, so errors do not
     compound over the 16 days;
  4. forecast zero for series that sold nothing in the last 56 days.

In [ ]:
import io, subprocess, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

COMP = 'store-sales-time-series-forecasting'
SEED = 42
H = 16                                       # forecast horizon (days)
START = pd.Timestamp('2013-01-01')           # day index 0
TRAIN_END = pd.Timestamp('2017-08-15')       # last day with known sales
TEST_START = pd.Timestamp('2017-08-16')      # a Wednesday
VAL_START = pd.Timestamp('2017-07-26')       # holdout window 26 Jul - 10 Aug, also starting on a Wednesday
FIRST_ORIGIN = pd.Timestamp('2015-01-07')    # first training snapshot (a Wednesday)
QUAKE = (pd.Timestamp('2016-04-16'), pd.Timestamp('2016-05-31'))   # earthquake and relief-donation period
ZERO_WINDOW = 56                             # series with no sales in this many days are forecast as 0

def day(ts):
    return (pd.Timestamp(ts) - START).days

# Plot styling: fixed colours, recessive grid
BLUE, ORANGE, AQUA, GREY = '#2a78d6', '#eb6834', '#1baf7a', '#9a9994'
plt.rcParams.update({
    'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'axes.edgecolor': GREY, 'axes.titleweight': 'bold', 'axes.titlesize': 11, 'font.size': 9,
    'lines.linewidth': 2,
})

## 1. Load data

In [ ]:
train = pd.read_csv('train.csv', parse_dates=['date'])
test = pd.read_csv('test.csv', parse_dates=['date'])
stores = pd.read_csv('stores.csv')
holidays = pd.read_csv('holidays_events.csv', parse_dates=['date'])
print('train', train.shape, '| test', test.shape)
print('train dates', train.date.min().date(), '->', train.date.max().date(),
      '| test dates', test.date.min().date(), '->', test.date.max().date())
train.head()

## 2. EDA
### 2.1 Total sales over time

In [ ]:
daily = train.groupby('date')['sales'].sum()
fig, ax = plt.subplots(figsize=(14, 3.6))
ax.plot(daily.index, daily.values / 1e3, color=BLUE, lw=0.6, alpha=0.45, label='daily')
ax.plot(daily.index, daily.rolling(28, center=True).mean().values / 1e3, color=BLUE, label='28-day mean')
ax.axvspan(*QUAKE, color=ORANGE, alpha=0.15, label='2016 earthquake period')
ax.set_title('Total daily sales, all stores (thousands)'); ax.legend(frameon=False, loc='upper left')
plt.show()

all_days = set(pd.date_range(START, TRAIN_END).strftime('%Y-%m-%d'))
print('Dates missing from train:', sorted(all_days - set(daily.index.strftime('%Y-%m-%d'))))
print('Days with under 10% of median sales:', list(daily[daily < 0.1 * daily.median()].index.strftime('%Y-%m-%d')))

### 2.2 Weekly and monthly cycles (last 12 months)

In [ ]:
recent = daily[daily.index > TRAIN_END - pd.Timedelta(days=365)]
recent = recent[recent > 0.1 * recent.median()]          # drop store-closure days
fig, axes = plt.subplots(1, 2, figsize=(14, 3.2))
dow = recent.groupby(recent.index.dayofweek).mean() / recent.mean()
axes[0].bar(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], dow.values, color=BLUE, width=0.6)
axes[0].axhline(1, color=GREY, lw=1, ls='--'); axes[0].set_title('Sales by weekday (relative to mean)')
dom = recent.groupby(recent.index.day).mean() / recent.mean()
axes[1].bar(dom.index, dom.values, color=[ORANGE if d in (1, 15, 16, 30, 31) else BLUE for d in dom.index], width=0.7)
axes[1].axhline(1, color=GREY, lw=1, ls='--'); axes[1].set_title('Sales by day of month (paydays on the 15th and month end)')
plt.tight_layout(); plt.show()

### 2.3 Promotions
Promoted rows sell far more, but promotions go to families that sell a lot anyway. Comparing each series with
itself (mean `log1p(sales)` on promoted days minus non-promoted days, 2017 only) isolates the lift.

In [ ]:
r = train[train.date >= '2017-01-01'].assign(promo=lambda d: d.onpromotion > 0, log_sales=lambda d: np.log1p(d.sales))
within = r.groupby(['store_nbr', 'family', 'promo'])['log_sales'].mean().unstack().dropna()
lift = np.expm1((within[True] - within[False]).groupby(level='family').median()).sort_values()
fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(lift.index, lift.values * 100, color=BLUE, height=0.6)
ax.set_title('Median within-series sales lift on promoted days, 2017 (%)')
plt.tight_layout(); plt.show()
print(f'{(train.onpromotion > 0).mean():.1%} of training rows have promotions, '
      f'{(test.onpromotion > 0).mean():.1%} of test rows')

## 3. Series × day matrices
Everything below works on two matrices with one row per (store, family) series and one column per day from
1 January 2013 to 31 August 2017:
- `S`: `log1p(sales)`. Christmas Day has no rows (stores closed) and is filled with 0; the test days are NaN.
- `P`: number of promoted items, known for the test days as well.

Store totals and family averages (over the 54 stores) are kept as extra matrices, giving each series the
context of its store and its product family.

In [ ]:
DATES = pd.date_range(START, test.date.max())
N_DAYS = len(DATES)
D_TEST, D_VAL, D_FIRST = day(TEST_START), day(VAL_START), day(FIRST_ORIGIN)

series = train[['store_nbr', 'family']].drop_duplicates().sort_values(['store_nbr', 'family']).reset_index(drop=True)
SERIES = pd.MultiIndex.from_frame(series)
N_SERIES = len(series)

def to_matrix(df, col):
    """Long (date, store, family) rows -> float32 matrix [series, day]; days without a row stay 0."""
    M = np.zeros((N_SERIES, N_DAYS), np.float32)
    rows = SERIES.get_indexer(pd.MultiIndex.from_frame(df[['store_nbr', 'family']]))
    M[rows, (df['date'] - START).dt.days.values] = df[col].values
    return M

S = np.log1p(to_matrix(train, 'sales'))
S[:, D_TEST:] = np.nan
P = to_matrix(pd.concat([train, test]), 'onpromotion')

st = stores.set_index('store_nbr').loc[np.sort(series.store_nbr.unique())]
st_idx = pd.Index(st.index).get_indexer(series.store_nbr)
fam_idx = pd.Index(np.sort(series.family.unique())).get_indexer(series.family)

sales_raw = pd.DataFrame(np.expm1(S[:, :D_TEST]))
store_total = sales_raw.groupby(st_idx).sum().values                                    # [store, day]
STORE_AGG = np.log1p(store_total)[st_idx].astype(np.float32)                           # each series' store total
FAM_AGG = np.log1p(sales_raw.groupby(fam_idx).mean().values)[fam_idx].astype(np.float32)  # family mean over stores
print('S', S.shape, '| P', P.shape)

### 3.1 Dead and late-starting series

In [ ]:
never = S[:, :D_TEST].sum(1) == 0
recent_zero = S[:, D_TEST - ZERO_WINDOW:D_TEST].sum(1) == 0
print(f'{never.sum()} series never sold anything; {recent_zero.sum()} sold nothing in the last {ZERO_WINDOW} days '
      '(these are forecast as 0)')
display(series[recent_zero].groupby('family').size().sort_values(ascending=False).head(8)
        .rename(f'series with no sales in the last {ZERO_WINDOW} days').to_frame())

store_first = pd.Series(DATES[(store_total > 0).argmax(1)], index=st.index, name='first day with sales')
print('Stores that opened after January 2013:')
display(store_first[store_first > '2013-01-31'].dt.date.to_frame())

## 4. Calendar
Holidays are matched to stores the way the data defines them: national ones apply everywhere, regional ones to
stores in that state, local ones to stores in that city. Holidays marked `transferred` were moved to another day
(listed as type `Transfer`) and are treated as normal days. `Bridge` days are extra days off; `Work Day` rows
are Saturdays worked to make up for them.

In [ ]:
def day_flags(dates):
    f = np.zeros(N_DAYS, np.float32)
    d = (pd.to_datetime(pd.Series(dates)) - START).dt.days.values
    f[d[(d >= 0) & (d < N_DAYS)]] = 1
    return f

hol = holidays[~holidays['transferred']]
nat = hol[hol.locale == 'National']
NAT_HOLIDAY = day_flags(nat.loc[nat.type.isin(['Holiday', 'Transfer', 'Bridge']), 'date'])
NAT_ADDITIONAL = day_flags(nat.loc[nat.type == 'Additional', 'date'])   # days around Christmas, Mother's Day, ...
NAT_EVENT = day_flags(nat.loc[nat.type == 'Event', 'date'])             # e.g. sporting events, earthquake aftermath
WORK_DAY = day_flags(nat.loc[nat.type == 'Work Day', 'date'])
REG_HOLIDAY = np.stack([day_flags(hol.loc[(hol.locale == 'Regional') & (hol.locale_name == s), 'date'])
                        for s in st.state])[st_idx]
LOC_HOLIDAY = np.stack([day_flags(hol.loc[(hol.locale == 'Local') & (hol.locale_name == c), 'date'])
                        for c in st.city])[st_idx]

STATIC = pd.DataFrame({
    'store_nbr': st_idx, 'family': fam_idx,
    'store_type': st['type'].astype('category').cat.codes.values[st_idx],
    'cluster': st['cluster'].values[st_idx],
    'city': st['city'].astype('category').cat.codes.values[st_idx],
    'state': st['state'].astype('category').cat.codes.values[st_idx],
})
CATS = list(STATIC.columns)
print('Holidays and events in the test window:')
holidays[(holidays.date >= TEST_START) & (holidays.date <= test.date.max())]

## 5. Features
For a snapshot taken at origin day `t` (the first day to forecast), every feature uses only days before `t`,
except promotions and the calendar, which are known in advance. All sales features are on the `log1p` scale.

| Block | Features |
|---|---|
| Recent level | Mean over the last 3, 7, 14, 28, 56, 112 and 224 days; median, standard deviation and share of zero days for windows of 7+ days; exponentially weighted mean (decay 0.9) |
| Last week | Each of the last 7 days individually |
| Weekly profile | For each weekday, the mean of its last 4, 12 and 26 occurrences; the family-level weekday mean over 8 weeks |
| Context | Store total and family average over the same windows |
| Promotions | Promotion intensity in the past windows; promoted items on each of the 16 forecast days and their sum; mean sales on promoted vs non-promoted days over the last 112 days |
| Life cycle | Days since the last sale |
| Static | Store, family, store type, cluster, city, state (LightGBM categoricals) |
| Target day (per horizon) | Promotions that day and vs the 28-day average, the weekday profile of that weekday, day of month, payday, national holiday / additional day / event / work day, regional and local holiday, national holiday the day before or after |

In [ ]:
def base_features(t):
    """Features shared by all 16 horizons for a snapshot at origin day t."""
    f = {}
    for w in [3, 7, 14, 28, 56, 112, 224]:
        x = S[:, t - w:t]
        f[f'mean_{w}'] = x.mean(1)
        if w >= 7:
            f[f'median_{w}'] = np.median(x, 1)
            f[f'std_{w}'] = x.std(1)
            f[f'zero_share_{w}'] = (x == 0).mean(1)
            f[f'promo_{w}'] = P[:, t - w:t].mean(1)
            f[f'store_mean_{w}'] = STORE_AGG[:, t - w:t].mean(1)
            f[f'family_mean_{w}'] = FAM_AGG[:, t - w:t].mean(1)
    decay = 0.9 ** np.arange(56)[::-1]
    f['ewm_56'] = S[:, t - 56:t] @ decay / decay.sum()
    for d in range(1, 8):
        f[f'lag_{d}'] = S[:, t - d]
    for k in range(7):                       # k = 0 is the origin's own weekday
        for n in [4, 12, 26]:
            f[f'dow{k}_mean_{n}'] = S[:, t - 7 + k - 7 * np.arange(n)].mean(1)
        f[f'family_dow{k}_mean_8'] = FAM_AGG[:, t - 7 + k - 7 * np.arange(8)].mean(1)
    for j in range(H):
        f[f'promo_day_{j + 1}'] = P[:, t + j]
    f['promo_next_16'] = P[:, t:t + H].sum(1)
    sold = S[:, t - 365:t] > 0
    f['days_since_sale'] = np.where(sold.any(1), sold[:, ::-1].argmax(1), 365)
    x, promo = S[:, t - 112:t], P[:, t - 112:t] > 0
    f['mean_on_promo_days'] = np.where(promo.any(1), (x * promo).sum(1) / np.maximum(promo.sum(1), 1), np.nan)
    f['mean_off_promo_days'] = np.where((~promo).any(1), (x * ~promo).sum(1) / np.maximum((~promo).sum(1), 1), np.nan)
    return pd.concat([STATIC, pd.DataFrame(f)], axis=1)

def horizon_features(t, h):
    """Base features plus everything about the target day t + h (h = 0 is the first forecast day)."""
    d = t + h
    X = BASE[t].copy()
    X['promo_target'] = P[:, d]
    X['promo_target_vs_28'] = np.log1p(P[:, d]) - np.log1p(P[:, t - 28:t].mean(1))
    X['dow_target_mean_4'] = X[f'dow{h % 7}_mean_4']      # day t+h falls on the same weekday as day t-7+(h%7)
    X['dow_target_mean_12'] = X[f'dow{h % 7}_mean_12']
    X['dow_target_vs_mean_56'] = X['dow_target_mean_12'] - X['mean_56']
    X['day_of_month'] = DATES[d].day
    X['payday'] = float(DATES[d].day == 15 or DATES[d].is_month_end)
    X['nat_holiday'] = NAT_HOLIDAY[d]
    X['nat_additional'] = NAT_ADDITIONAL[d]
    X['nat_event'] = NAT_EVENT[d]
    X['work_day'] = WORK_DAY[d]
    X['reg_holiday'] = REG_HOLIDAY[:, d]
    X['loc_holiday'] = LOC_HOLIDAY[:, d]
    X['nat_holiday_adjacent'] = NAT_HOLIDAY[d - 1] + NAT_HOLIDAY[min(d + 1, N_DAYS - 1)]
    return X

ORIGINS = np.arange(D_FIRST, D_TEST + 1, 7)               # every Wednesday from 2015-01-07 to the test start
assert D_TEST in ORIGINS and D_VAL in ORIGINS and DATES[D_TEST].dayofweek == 2
BASE = {t: base_features(t) for t in ORIGINS}
ACTIVE = {t: S[:, t - ZERO_WINDOW:t].sum(1) > 0 for t in ORIGINS}
QUAKE_D = (day(QUAKE[0]), day(QUAKE[1]))

def make_dataset(h, cutoff):
    """Training rows for horizon h: every snapshot whose target day is before `cutoff` and outside the earthquake
    period. Series that sold nothing in the 56 days before the snapshot are left out (they are forecast as 0)."""
    Xs, ys = [], []
    for t in ORIGINS:
        d = t + h
        if d >= cutoff or QUAKE_D[0] <= d <= QUAKE_D[1]:
            continue
        a = ACTIVE[t]
        Xs.append(horizon_features(t, h)[a])
        ys.append(S[a, d])
    return pd.concat(Xs, ignore_index=True), np.concatenate(ys)

Xs, ys = make_dataset(0, D_VAL)
print(f'{len(ORIGINS)} snapshots | horizon-1 training set {Xs.shape} | {Xs.shape[1]} features')
del Xs, ys

## 6. Validation
The holdout is the last complete 16-day window that starts on a Wednesday: 26 July to 10 August 2017. Each
horizon's model trains on every snapshot whose target day falls before 26 July, so the short horizons also learn
from the most recent weeks. Early stopping on the holdout picks the number of trees per horizon (which makes the
holdout score slightly optimistic); the final models are then refitted on all data up to 15 August.

Two baselines show what the model adds: the mean of the last 14 days, and the mean of the same weekday over the last
4 weeks.

In [ ]:
PARAMS = dict(objective='regression', metric='rmse', learning_rate=0.05, num_leaves=63, min_data_in_leaf=100,
              feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, max_bin=127,
              cat_smooth=10, verbose=-1, n_jobs=-1, seed=SEED)

Y_VAL = S[:, D_VAL:D_VAL + H]
act_val = ACTIVE[D_VAL]
val_pred = np.zeros((N_SERIES, H), np.float32)
best_iter, horizon_rmsle, importances = [], [], []
t0 = time.time()
for h in range(H):
    Xtr, ytr = make_dataset(h, D_VAL)
    Xva = horizon_features(D_VAL, h)
    dtr = lgb.Dataset(Xtr, ytr, categorical_feature=CATS)
    dva = lgb.Dataset(Xva[act_val], Y_VAL[act_val, h], categorical_feature=CATS, reference=dtr)
    m = lgb.train(PARAMS, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    val_pred[:, h] = np.where(act_val, np.clip(m.predict(Xva, num_iteration=m.best_iteration), 0, None), 0)
    best_iter.append(m.best_iteration)
    horizon_rmsle.append(np.sqrt(np.mean((val_pred[:, h] - Y_VAL[:, h]) ** 2)))
    importances.append(pd.Series(m.feature_importance('gain'), m.feature_name()))
    print(f'day {h + 1:2d} ({DATES[D_VAL + h]:%a %d %b}): {len(ytr):,} rows, {m.best_iteration:4d} trees, '
          f'RMSLE {horizon_rmsle[-1]:.4f}  [{time.time() - t0:.0f}s]')

def rmsle(pred):
    return np.sqrt(np.mean((pred - Y_VAL) ** 2))

base_val = BASE[D_VAL]
baselines = {
    'Mean of last 14 days': np.repeat(base_val['mean_14'].values[:, None], H, 1),
    'Same weekday, last 4 weeks': np.stack([base_val[f'dow{h % 7}_mean_4'].values for h in range(H)], 1),
    'LightGBM, one model per day': val_pred,
}
display(pd.Series({k: rmsle(v) for k, v in baselines.items()}, name='holdout RMSLE').round(4).to_frame())

### 6.1 Where the errors are

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.4), gridspec_kw={'width_ratios': [1, 1.4]})
axes[0].bar(np.arange(1, H + 1), horizon_rmsle, color=BLUE, width=0.6)
axes[0].set_title('Holdout RMSLE by forecast day'); axes[0].set_xlabel('days ahead')
hist = slice(D_VAL - 28, D_VAL)
axes[1].plot(DATES[hist], np.expm1(S[:, hist]).sum(0) / 1e3, color=GREY, label='history')
axes[1].plot(DATES[D_VAL:D_VAL + H], np.expm1(Y_VAL).sum(0) / 1e3, color=BLUE, label='actual')
axes[1].plot(DATES[D_VAL:D_VAL + H], np.expm1(val_pred).sum(0) / 1e3, color=ORANGE, ls='--', label='forecast')
axes[1].set_title('Total daily sales in the holdout window (thousands)'); axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()

err = pd.DataFrame((val_pred - Y_VAL) ** 2).mean(1)
display(err.groupby(series.family.values).mean().pow(0.5).sort_values(ascending=False).head(10)
        .rename('holdout RMSLE').round(4).to_frame())

### 6.2 Feature importance (gain, averaged over the 16 models)

In [ ]:
imp = pd.concat(importances, axis=1).fillna(0)
imp = (imp / imp.sum()).mean(axis=1).sort_values().tail(25)
fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(imp.index, imp.values, color=BLUE, height=0.6)
ax.set_title('LightGBM gain importance, top 25 (share)')
plt.tight_layout(); plt.show()

## 7. Refit and forecast
Each horizon is refitted on every snapshot whose target day is on or before 15 August, with 10% more trees than
early stopping chose (the training set grows by the 2-3 most recent weeks).

In [ ]:
test_pred = np.zeros((N_SERIES, H), np.float32)
act_test = ACTIVE[D_TEST]
t0 = time.time()
for h in range(H):
    Xtr, ytr = make_dataset(h, D_TEST)
    m = lgb.train(PARAMS, lgb.Dataset(Xtr, ytr, categorical_feature=CATS), int(best_iter[h] * 1.1) + 10)
    test_pred[:, h] = np.where(act_test, np.clip(m.predict(horizon_features(D_TEST, h)), 0, None), 0)
    print(f'day {h + 1:2d}: {len(ytr):,} rows  [{time.time() - t0:.0f}s]')

fig, ax = plt.subplots(figsize=(14, 3.4))
hist = slice(D_TEST - 56, D_TEST)
ax.plot(DATES[hist], np.expm1(S[:, hist]).sum(0) / 1e3, color=BLUE, label='actual')
ax.plot(DATES[D_TEST:D_TEST + H], np.expm1(test_pred).sum(0) / 1e3, color=ORANGE, ls='--', label='forecast')
ax.set_title('Total daily sales: last 8 weeks and the 16-day forecast (thousands)'); ax.legend(frameon=False)
plt.show()

## 8. Submission

In [ ]:
sample = pd.read_csv('sample_submission.csv')
rows = SERIES.get_indexer(pd.MultiIndex.from_frame(test[['store_nbr', 'family']]))
cols = (test['date'] - TEST_START).dt.days.values
sub = pd.DataFrame({'id': test['id'], 'sales': np.expm1(test_pred[rows, cols])})
assert (rows >= 0).all() and (sub['id'].values == sample['id'].values).all()
sub.to_csv('submission.csv', index=False)
print(sub.shape, f'| {(sub.sales == 0).mean():.1%} zero forecasts')
sub.describe()

In [ ]:
def kaggle_cli(*args):
    # The CLI prints a UTF-8 progress bar; Windows' default cp1252 decoding crashes on it
    return subprocess.run(['kaggle', 'competitions', *args], capture_output=True, text=True,
                          encoding='utf-8', errors='replace')

# Accept the competition rules on Kaggle first, otherwise the CLI is refused (400 Bad Request)
SUBMIT = False   # set to True to submit
msg = f'LightGBM, one model per forecast day | holdout RMSLE {rmsle(val_pred):.5f}'
if SUBMIT:
    out = kaggle_cli('submit', COMP, '-f', 'submission.csv', '-m', msg)
    print(out.stdout[-2000:], out.stderr[-300:])

In [ ]:
# Wait for scoring. Getting Started competitions only have a public leaderboard.
if SUBMIT:
    for _ in range(30):
        subs = pd.read_csv(io.StringIO(kaggle_cli('submissions', '-c', COMP, '-v').stdout))
        if 'pending' not in str(subs.loc[0, 'status']).lower():
            break
        time.sleep(20)
    display(subs.head(3))